# W2 Project Starter · When Local Messages Mislead

**Total: 10 points.** Read the [full specification and rubric](https://github.com/lunalab-ai/advML/blob/main/course/notion/w2/w2-project.md). The deadline is shown in SmartClass.

This starter automatically prepares the course module and demonstrates the output schema. Assessed methods are intentionally unfinished. All default cells run, but a successful default run does not mean the project is complete. Use a CPU runtime and save your own copy.


## Shared API: read before calling

[Complete API reference](https://github.com/lunalab-ai/advML/blob/2026-fall-w2-explained/src/API.md)

### [PairwiseModel · definition](https://github.com/lunalab-ai/advML/blob/2026-fall-w2-explained/src/advml_pgm.py#L43)

Finite undirected model represented by log potentials, not fitted parameters.
log_unary: numeric (N,K) array, N nodes and K states per node, N,K >= 1.
log_edges: dict {(u,v): (K,K) numeric array}, indexed [state_u,state_v].
Node IDs are 0..N-1 and states are 0..K-1. Reverse keys are transposed
to canonical order; duplicate undirected edges and self edges are invalid.
The unnormalized log joint sums one unary entry per node and one edge entry
per edge. Factors need not be normalized probability tables. Negative infinity
encodes a hard zero; NaN and positive infinity raise ValueError. Arrays are
copied at construction. Methods perform inference/conditioning; no training
data or fit method is involved. Example: PairwiseModel(np.zeros((2,2)),
{(0,1):np.log([[2.,1.],[1.,2.]])}).

### [PairwiseModel.edge · definition](https://github.com/lunalab-ai/advML/blob/2026-fall-w2-explained/src/advml_pgm.py#L98)

Return the log edge table oriented from node u to node v.
u,v: distinct node IDs of an existing edge. Output shape (K,K), with u
states in rows and v states in columns. Missing edges raise KeyError.
The returned array may share storage with the model: treat it as read-only.
edge(1,0) is edge(0,1).T, not a different undirected factor.

### [PairwiseModel.condition · definition](https://github.com/lunalab-ai/advML/blob/2026-fall-w2-explained/src/advml_pgm.py#L106)

Return a new model clamped to evidence without changing this model.
evidence: dict {node_id: observed_state}, IDs in 0..N-1, states in 0..K-1.
Invalid indices raise ValueError. Inconsistent total mass is detected by
the inference routine. Nonselected unary states receive -inf; the selected
state keeps its original scale. The graph and N remain unchanged: clamping
a cycle does not structurally turn it into a tree. Evidence probability is
exp(log_z_clamped - log_z_original), not exp(log_z_clamped) in general.

### [enumerate_exact · definition](https://github.com/lunalab-ai/advML/blob/2026-fall-w2-explained/src/advml_pgm.py#L124)

Enumerate the complete joint as an independent small-model oracle.
model: PairwiseModel. max_states: maximum K**N assignments, default 262144.
Returns dict: marginals (N,K) normalized probabilities; log_z float;
states (K**N,N) integer assignments; probabilities (K**N,) normalized
joint weights in corresponding order. No input is mutated. Excess states
or zero total mass raise ValueError. Runtime O(K**N*(N+E)); joint storage
is exponential. Use for verification, not large graphs. For clamped models,
log_z retains unnormalized evidence mass and is not generally log P(e).

### [tree_sum_product · definition](https://github.com/lunalab-ai/advML/blob/2026-fall-w2-explained/src/advml_pgm.py#L149)

Run exact two-pass log-domain sum-product on a connected tree.
model: PairwiseModel (single node allowed). root: starting node, default 0.
Returns dict marginals (N,K), log_z float, and log_messages dict whose
(u,v) entry has K values indexed by the receiver v's state. Messages
retain scale so log_z is recoverable. Model is not changed. Invalid root,
a cycle/disconnected graph, or zero total mass raises ValueError.
Time O(N*K**2) at bounded degree, with O(K*sum(degree**2)) extra work
from explicitly recomputing neighboring products. Different roots should
produce the same marginals up to floating-point tolerance.

### [loopy_sum_product · definition](https://github.com/lunalab-ai/advML/blob/2026-fall-w2-explained/src/advml_pgm.py#L202)

Run synchronous approximate sum-product with probability-space damping.
model: PairwiseModel. damping: OLD-message weight in [0,1), default 0.
max_iter: positive iteration limit, default 300. tol: positive stopping
threshold on maximum elementwise normalized-message change, default 1e-10.
Returns dict marginals (N,K), residuals (iterations,), converged bool,
and iterations int. Inputs are unchanged; messages start uniformly.
Each update uses the previous iteration, not partially updated neighbors.
No log_z estimate is claimed. A small residual is not a bound on marginal
error on cycles. Invalid settings/impossible local mass raise ValueError.
Compare accuracy with enumerate_exact only on tractable models.

### [ising_model · definition](https://github.com/lunalab-ai/advML/blob/2026-fall-w2-explained/src/advml_pgm.py#L246)

Construct a binary PairwiseModel from fields and couplings.
fields: length-N numeric array of local fields h_i. couplings: dict
{(u,v): J_uv} of scalar edge strengths. State 0 maps to spin -1 and
state 1 to +1. Unary log potential is h_i*s_i; edge log potential is
J_uv*s_u*s_v, so positive J favors agreement. Returns a new model;
no fitting or random sampling occurs. Example: ising_model(np.zeros(2),
{(0,1):0.8}) favors equal states without a local preference.

### [max_marginal_tv · definition](https://github.com/lunalab-ai/advML/blob/2026-fall-w2-explained/src/advml_pgm.py#L259)

Return max_i 0.5*sum_k(abs(p[i,k]-q[i,k])) as a float.
p,q: matching normalized probability arrays of shape (N,K). The caller
is responsible for nonnegative, unit-sum rows. Shape mismatch raises
ValueError. Inputs are not changed. This compares single-node marginals,
not joint distributions and not consecutive messages. Example: rows
[0.2,0.8] versus [0.5,0.5] have TV 0.3.


In [ ]:
from pathlib import Path
import hashlib, importlib.util, sys, tempfile, urllib.request, subprocess
for package, version in [("numpy", "2.2.6"), ("matplotlib", "3.10.3")]:
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", f"{package}=={version}"])
CODE_REF = "2026-fall-w2-explained"
EXPECTED_SHA256 = "97c68026a1024694ec97a632449aa6ada97d18fe0a8fb94212d260818fbf0a28"
CODE_URL = f"https://raw.githubusercontent.com/lunalab-ai/advML/{CODE_REF}/src/advml_pgm.py"
local = Path("src/advml_pgm.py")
if local.is_file() and hashlib.sha256(local.read_bytes()).hexdigest() == EXPECTED_SHA256:
    module_path = local.resolve()
    print("Using verified course-checkout code.")
else:
    module_path = Path(tempfile.gettempdir()) / f"advml_pgm_{EXPECTED_SHA256[:16]}.py"
    if not module_path.is_file() or hashlib.sha256(module_path.read_bytes()).hexdigest() != EXPECTED_SHA256:
        for attempt in range(3):
            try:
                with urllib.request.urlopen(CODE_URL, timeout=30) as response:
                    payload = response.read()
                if hashlib.sha256(payload).hexdigest() != EXPECTED_SHA256:
                    raise RuntimeError("Course module checksum mismatch. Stop and report this; do not bypass the check.")
                module_path.write_bytes(payload)
                break
            except Exception as exc:
                if attempt == 2:
                    raise RuntimeError("Could not prepare the fixed course module. Check the connection and rerun this cell; the complete public course checkout is the offline alternative.") from exc
    print("Using verified fixed-revision public code (download/cache).")
spec = importlib.util.spec_from_file_location("advml_pgm", module_path)
pgm = importlib.util.module_from_spec(spec)
sys.modules["advml_pgm"] = pgm
spec.loader.exec_module(pgm)
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from advml_pgm import PairwiseModel, enumerate_exact, tree_sum_product, loopy_sum_product, ising_model, max_marginal_tv
print("Python", sys.version.split()[0], "NumPy", np.__version__, "Matplotlib", matplotlib.__version__)
print("Reference:", CODE_REF, "SHA-256:", EXPECTED_SHA256)
SEED = 20260908
rng = np.random.default_rng(SEED)


## Prediction ledger — complete before your search

Write your claim, mechanism, falsifying metric/tolerance, planned model families and version/timestamp here. Keep failed predictions. Distinguish tuning from held-out evaluation.


In [ ]:
prediction_ledger = {"claim": "", "mechanism": "", "falsification_rule": "", "frozen_method_revision": ""}
print("Complete the ledger before parameter sweeps:", prediction_ledger)


## Baseline and result schema

This four-node cycle is only a smoke test, not a sufficient study. Design your own fields, coupling regimes, frustrated cycle and tree controls.


In [ ]:
from time import perf_counter
model = ising_model(np.array([0.2, -0.1, 0.05, 0.0]), {(0,1):0.8,(1,2):0.8,(2,3):0.8,(0,3):-0.8})
exact = enumerate_exact(model)
started = perf_counter()
baseline = loopy_sum_product(model, damping=0.3, max_iter=1000, tol=1e-9)
example_result = {"seed": SEED, "family": "starter-only", "n": model.n,
 "method": "loopy-baseline", "converged": bool(baseline["converged"]),
 "residual": float(baseline["residuals"][-1]), "iterations": baseline["iterations"],
 "directed_message_updates": 2*len(model.log_edges)*baseline["iterations"],
 "max_node_tv": max_marginal_tv(exact["marginals"],baseline["marginals"]),
 "seconds": perf_counter()-started}
print(example_result)


## Your exact correction

Derive the reduced model, branch normalizers and reconstruction in Markdown before coding. Do not call enumeration inside your method. Use it only as an independent small-case checker. Explain why the `condition` utility alone is not graph reduction.


In [ ]:
def cycle_cutset_exact(model, cut_node):
    """Your implementation: return {'marginals': array(N,K), 'log_z': float}."""
    raise NotImplementedError("Implement the derived reduction and reconstruction.")

RUN_MY_METHOD = False  # switch on after implementing; not an automatic completion flag
if RUN_MY_METHOD:
    answer = cycle_cutset_exact(model, cut_node=0)
    np.testing.assert_allclose(answer["marginals"], exact["marginals"], atol=1e-8)
    np.testing.assert_allclose(answer["log_z"], exact["log_z"], atol=1e-8)
else:
    print("Assessed correction is unfinished. See specification for required tests.")


## Your extension, held-out tests and ablation

State your proposed mechanism and compute budget here. Freeze the method before evaluating five additional seeds and an unseen structure or parameter regime. Record every outcome and compare with a simpler control. Include a 60-node cycle scalability demonstration of your exact correction without attempting enumeration.


In [ ]:
development_seeds = [101, 102, 103]
held_out_seeds = [201, 202, 203, 204, 205]
assert not set(development_seeds) & set(held_out_seeds)
print("Separate tuning and evaluation. Add at least one unseen regime/structure.")


## Export a result file

Replace this one-row example with your complete results. Download the generated file from Colab's Files panel for submission. Include full model configurations or a deterministic configuration generator in your ZIP.


In [ ]:
import json
Path("results-example.json").write_text(json.dumps([example_result], indent=2), encoding="utf-8")
print("Wrote results-example.json. This is an example, not a completed project.")


## Written defense and submission checklist

Answer the three defense questions in the specification using your own equations, code and results. Submit report.pdf, executed project.ipynb, reusable implementation, README, results table, tests and assistance.md. Follow the existing integrity policy. No claim is made that a coding task can prevent all AI assistance; assessment focuses on demonstrated reasoning, original choices and validated evidence.


In [ ]:
np.testing.assert_allclose(exact["marginals"].sum(axis=1), 1)
assert baseline["iterations"] >= 1
